# AUGMENT DATA — Le Panier-Sûr

Génération de **N observations synthétiques par espèce** à partir du CSV nettoyé, pour alimenter un modèle de classification par nom.

- **Entrée** : `data/clean_train_data/champignons_clean.csv`
- **Sortie** : `data/augmented/champignons_augmented.csv`

## ⚠️ Mise en garde data science

Cette augmentation multiplie artificiellement les lignes. Elle fonctionne **uniquement si** :
1. Le split train/test est fait *après* augmentation et *stratifié par espèce* → le modèle voit chaque espèce à l'entraînement ET au test (mémorisation contrôlée, pas généralisation zero-shot).
2. Le bruit introduit est **léger et réaliste** — pas de nouvelles features, uniquement des features existantes qu'on peut "manquer d'observer".
3. Tu n'espères pas que le modèle reconnaisse une espèce jamais vue. Pour ça, il faut de vraies données supplémentaires (multi-sources).

**Ce que l'augmentation apporte** : robustesse du modèle face à une observation partielle (pas toutes les couleurs vues, **une seule taille mesurée tirée uniformément dans l'intervalle [min, max] de l'espèce**, un seul mois d'observation…).

**Ce qu'elle n'apporte PAS** : de la vraie diversité d'espèces. 219 espèces × 100 = 21 900 lignes, mais toujours 219 classes.

> **Note** : on ne conserve plus `*_taille_min_cm` / `*_taille_max_cm` dans la sortie — l'utilisateur final mesure **une** taille, pas deux. Chaque ligne augmentée contient `chapeau_taille_cm` et `pied_taille_cm`, tirés uniformément dans la plage spécifique à l'espèce.

## 1. Chargement

In [10]:
import numpy as np
import pandas as pd

CLEAN_CSV = "../../data/clean_train_data/champignons_clean.csv"
df = pd.read_csv(CLEAN_CSV)
print(f"{len(df)} espèces, {len(df.columns)} colonnes")
df.head(2)

217 espèces, 165 colonnes


,nom,statut,a_un_chapeau,a_des_pores,a_des_lames,a_un_pied,a_de_la_chair,chapeau_couleur_blanc,chapeau_couleur_brun,chapeau_couleur_jaune,...,habitat_type_jardins,habitat_type_bois,habitat_type_chenes,habitat_type_hetres,habitat_type_pins,habitat_type_bouleaux,habitat_type_chataigniers,habitat_type_charmes,habitat_type_melezes,habitat_type_bois_morts
0,AGARIC AUGUSTE,Champignon à rejeter,1,0,1,1,1,1,1,1,...,0,1,0,0,0,0,0,0,0,0
1,AGARIC DES JACHÈRES,Les excellents champignons,1,0,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


## 2. Paramètres d'augmentation

| Paramètre | Rôle |
|---|---|
| `N_SAMPLES` | Nombre d'observations synthétiques par espèce |
| `BINARY_DROPOUT` | Probabilité qu'une feature binaire à 1 soit flippée à 0 (simule une observation incomplète) |
| `SEED` | Reproductibilité |

La taille est tirée **uniformément** dans `[min, max]` de l'espèce — pas de paramètre de variance à régler.

In [11]:
N_SAMPLES      = 1000
BINARY_DROPOUT = 0.00   # 5% des 1 deviennent 0 (dropout doux)
SEED           = 42

rng = np.random.default_rng(SEED)

## 3. Identification des colonnes par type

Le CSV augmenté ne contiendra que les **features numériques** + `nom` (label) + `statut` (label alternatif). Les colonnes texte brut (`chapeau`, `pores`, `lames`, `pied`, `chair`, `odeur`, `saveur`, `habitat`, `saison`) sont exclues.

Catégorisation :
- **Tailles source** (`*_taille_min_cm`, `*_taille_max_cm`) → lues dans le CSV clean, **collapsées en une seule colonne `*_taille_cm`** par tirage uniforme dans [min, max]
- **Saison binaire** (`saison_mois_01` … `saison_mois_12`) → tirer un seul mois parmi les actifs
- **Autres binaires** (`*_couleur_*`, `*_texture_*`, `a_un_*`, `habitat_type_*`…) → dropout léger
- **Labels** (`nom`, `statut`) → copie identique

In [12]:
size_src_cols = [c for c in df.columns if c.endswith("_taille_min_cm") or c.endswith("_taille_max_cm")]

# Paires (part, min_col, max_col) pour chaque "part" qui a bien min ET max
size_ranges = {}
for c in df.columns:
    if c.endswith("_taille_min_cm"):
        part = c[: -len("_taille_min_cm")]
        max_col = f"{part}_taille_max_cm"
        if max_col in df.columns:
            size_ranges[part] = (c, max_col)

# Colonnes de sortie : une seule par "part"
size_out_cols = [f"{part}_taille_cm" for part in size_ranges]

season_cols  = [c for c in df.columns if c.startswith("saison_mois_")]

# Colonnes binaires "classiques" : toutes les 0/1 sauf saison (traitée à part)
binary_cols = []
for c in df.columns:
    if c in season_cols or c in size_src_cols:
        continue
    if df[c].dropna().isin([0, 1]).all() and df[c].dtype != object:
        binary_cols.append(c)

# Seules colonnes non-numériques conservées : nom (label principal) et statut (label alternatif)
LABEL_COLS = ["nom", "statut"]
label_cols = [c for c in LABEL_COLS if c in df.columns]

# Toutes les autres colonnes texte (descriptions brutes) sont droppées
dropped_text_cols = [c for c in df.columns
                     if c not in size_src_cols + season_cols + binary_cols + label_cols]

print(f"Tailles source (min/max) : {len(size_src_cols)}  ->  {len(size_out_cols)} tailles uniques en sortie")
print(f"  {size_out_cols}")
print(f"Saison     : {len(season_cols)}")
print(f"Binaires   : {len(binary_cols)}")
print(f"Labels     : {label_cols}")
print(f"Droppées   : {dropped_text_cols}")

Tailles source (min/max) : 4  ->  2 tailles uniques en sortie
  ['chapeau_taille_cm', 'pied_taille_cm']
Saison     : 12
Binaires   : 147
Labels     : ['nom', 'statut']
Droppées   : []


## 4. Fonctions d'augmentation

### Taille — tirage uniforme

Pour chaque paire `(min, max)` de l'espèce, on tire `n` valeurs uniformément dans l'intervalle. L'utilisateur final fournira **une seule taille mesurée**, donc on simule exactement ça : un échantillon = une mesure plausible pour l'espèce.

Si l'intervalle est invalide (NaN ou max ≤ 0), on renvoie 0.

### Binaires — dropout

Chaque `1` est conservé avec probabilité `1 - BINARY_DROPOUT`, sinon flippé à `0`. Les `0` restent toujours `0` (on n'invente pas de features absentes de la description source).

### Saison — un mois observé

Au lieu de conserver les 12 flags, on tire un mois uniforme parmi les mois actifs. L'observation est alors datée de ce mois uniquement. On reconstruit ensuite les 12 colonnes avec un seul `1`.

In [13]:
def sample_size(vmin, vmax, n, rng):
    """n tirages uniformes dans [vmin, vmax]. Renvoie 0 si plage invalide."""
    if pd.isna(vmin) or pd.isna(vmax) or vmax <= 0 or vmax < vmin:
        return np.zeros(n)
    return rng.uniform(float(vmin), float(vmax), n)

def dropout_binary(values, n, rng):
    """Pour chaque valeur binaire, réplique n fois avec dropout sur les 1."""
    out = np.tile(values, (n, 1))            # shape (n, n_cols)
    mask_ones = out == 1
    flip = rng.random(out.shape) < BINARY_DROPOUT
    out[mask_ones & flip] = 0
    return out

def sample_season_month(active_months, n, rng):
    """Tire n mois parmi les actifs. Renvoie un array (n, 12) avec un seul 1 par ligne."""
    result = np.zeros((n, 12), dtype=int)
    if len(active_months) == 0:
        return result
    picks = rng.choice(active_months, size=n)  # valeurs dans 1..12
    result[np.arange(n), picks - 1] = 1
    return result

## 5. Boucle d'augmentation

Pour chaque espèce du dataset, on génère `N_SAMPLES` lignes en appliquant les trois transformations.

In [14]:
augmented_rows = []

for _, row in df.iterrows():
    # 1. Labels : recopie à l'identique (nom, statut)
    base = {col: [row[col]] * N_SAMPLES for col in label_cols}

    # 2. Tailles : tirage uniforme dans [min, max] -> UNE seule colonne par "part"
    sizes = {}
    for part, (cmin, cmax) in size_ranges.items():
        sizes[f"{part}_taille_cm"] = sample_size(row[cmin], row[cmax], N_SAMPLES, rng)

    # 3. Binaires : dropout
    bin_values = row[binary_cols].to_numpy().astype(int)
    bin_aug = dropout_binary(bin_values, N_SAMPLES, rng)
    bin_dict = {col: bin_aug[:, i] for i, col in enumerate(binary_cols)}

    # 4. Saison : un seul mois tiré parmi les actifs
    active = np.array([i + 1 for i, c in enumerate(season_cols) if row[c] == 1])
    season_aug = sample_season_month(active, N_SAMPLES, rng)
    season_dict = {col: season_aug[:, i] for i, col in enumerate(season_cols)}

    # Assemblage
    chunk = pd.DataFrame({**base, **sizes, **bin_dict, **season_dict})
    augmented_rows.append(chunk)

# Ordre des colonnes : labels d'abord, puis tailles (collapsées), puis binaires, puis saison
ordered_cols = label_cols + size_out_cols + binary_cols + season_cols
df_aug = pd.concat(augmented_rows, ignore_index=True)[ordered_cols]

print(f"Avant : {len(df)} espèces × {len(df.columns)} colonnes")
print(f"Après : {len(df_aug)} lignes × {len(df_aug.columns)} colonnes ({N_SAMPLES}× par espèce)")
df_aug.head(3)

Avant : 217 espèces × 165 colonnes
Après : 217000 lignes × 163 colonnes (1000× par espèce)


,nom,statut,chapeau_taille_cm,pied_taille_cm,a_un_chapeau,a_des_pores,a_des_lames,a_un_pied,a_de_la_chair,chapeau_couleur_blanc,...,saison_mois_03,saison_mois_04,saison_mois_05,saison_mois_06,saison_mois_07,saison_mois_08,saison_mois_09,saison_mois_10,saison_mois_11,saison_mois_12
0,AGARIC AUGUSTE,Champignon à rejeter,20.479121,6.868883,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
1,AGARIC AUGUSTE,Champignon à rejeter,13.777569,12.415669,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
2,AGARIC AUGUSTE,Champignon à rejeter,22.171958,7.806421,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0


## 6. Sanity checks

Validation rapide que l'augmentation a fait ce qu'on attend.

In [15]:
# (a) Chaque espèce a bien N_SAMPLES lignes
counts = df_aug["nom"].value_counts()
print(f"Lignes par espèce : min={counts.min()}, max={counts.max()} (attendu : {N_SAMPLES})")

# (b) Les tailles varient au sein d'une espèce et restent dans [min, max] source
sample_species = df_aug["nom"].iloc[0]
src_row = df[df["nom"] == sample_species].iloc[0]
print(f"\nEspèce témoin : {sample_species}")
for part, (cmin, cmax) in size_ranges.items():
    out_col = f"{part}_taille_cm"
    serie = df_aug.loc[df_aug["nom"] == sample_species, out_col]
    src_min, src_max = src_row[cmin], src_row[cmax]
    print(f"  {out_col}: observé [{serie.min():.2f}, {serie.max():.2f}]  "
          f"mean={serie.mean():.2f}  std={serie.std():.2f}  "
          f"(source [{src_min}, {src_max}])")

# (c) La saison ne contient qu'un seul 1 par ligne
if season_cols:
    season_sum = df_aug[season_cols].sum(axis=1)
    print(f"\nSaison — nb de mois actifs par ligne : unique={sorted(season_sum.unique())}")

# (d) Taux moyen de 1 avant/après sur les binaires (doit avoir légèrement baissé)
if binary_cols:
    rate_before = df[binary_cols].mean().mean()
    rate_after = df_aug[binary_cols].mean().mean()
    print(f"\nBinaires — taux moyen de 1 : {rate_before:.3f} → {rate_after:.3f} (dropout {BINARY_DROPOUT})")

Lignes par espèce : min=1000, max=1000 (attendu : 1000)

Espèce témoin : AGARIC AUGUSTE
  chapeau_taille_cm: observé [5.02, 24.98]  mean=14.94  std=5.83  (source [5.0, 25.0])
  pied_taille_cm: observé [6.01, 19.99]  mean=13.12  std=4.05  (source [6.0, 20.0])

Saison — nb de mois actifs par ligne : unique=[np.int64(0), np.int64(1)]

Binaires — taux moyen de 1 : 0.113 → 0.113 (dropout 0.0)


## 7. Export

Les observations sont **conservées groupées par espèce** (pas de shuffle) — chaque espèce occupe `N_SAMPLES` lignes consécutives. Le shuffle se fera au moment du `train_test_split` côté modélisation.

In [16]:
import os

OUT_DIR = "../../data/augmented"
OUT_DIR2 = "../../data/clean_train_data"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/champignons_augmented.csv"
OUT_CSV = f"{OUT_DIR2}/champignons_augmented.csv"

df_aug.to_csv(OUT_CSV, index=False)
print(f"{len(df_aug)} lignes, {len(df_aug.columns)} colonnes → {OUT_CSV}")
print(f"Ordre conservé : {N_SAMPLES} lignes consécutives par espèce.")

217000 lignes, 163 colonnes → ../../data/clean_train_data/champignons_augmented.csv
Ordre conservé : 1000 lignes consécutives par espèce.
